(sequence-search-sort)=

# Searching and Sorting Sequences


```{contents}
:local:
:depth: 2
```


A service desk needs to locate an ID and produce an ordered report. Choose the result contract first: any match, first match, or every match. Sorting creates a new order; indexes then refer to that order rather than original arrival positions. The examples use arrays with constant-cost indexed access.


## First-Match Linear Search

```{index} First-Match Linear Search
```

Linear search accepts unsorted data and returns the smallest matching index or −1. Its loop invariant says that the processed prefix contains no match. Empty input returns −1 without accessing an item. One equality call is counted as one probe.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] amounts={200,100,200,300};
foreach(int target in new[]{200,300,999})
{
 int index=ReportSearch.LinearFirst(amounts,target,EqualityComparer<int>.Default,out long probes);
 Console.WriteLine($"target={target}, first={index}, probes={probes}");
}
Console.WriteLine($"empty={ReportSearch.LinearFirst(Array.Empty<int>(),200,EqualityComparer<int>.Default,out _)}");
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


On a nonempty input, the best case is Θ(1) probes and the worst is Θ(n). A target at index k costs k+1 probes. Missing targets cost n. Finding every match requires continuing the scan and collecting indexes; a first-match interface cannot return all duplicates.

(binarysearching)=

## Iterative Binary Search

```{index} Iterative Binary Search
```

Binary search requires random access and an array sorted under the same comparer used for searching. The contract here returns any matching index or −1; duplicates do not guarantee a first match. The half-open interval [low,high) contains all remaining candidates.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,200,200,300,400};
foreach(int target in new[]{100,200,400,250})
{
 int index=ReportSearch.BinaryAny(sorted,target,Comparer<int>.Default,out long probes);
 Console.WriteLine($"target={target}, any={index}, probes={probes}");
}
Console.WriteLine($"empty={ReportSearch.BinaryAny(Array.Empty<int>(),100,Comparer<int>.Default,out _)}");
public static class ReportSearch
{
    public static int LinearFirst<T>(T[] values,T target,IEqualityComparer<T> equality,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(equality);
        probes=0;
        for(int i=0;i<values.Length;i++)
        {
            probes++;
            if(equality.Equals(values[i],target))return i;
        }
        return -1;
    }
    // Sorted under the same comparer. Returns any match, or -1.
    public static int BinaryAny<T>(T[] values,T target,IComparer<T> comparer,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order==0)return middle;
            if(order<0)low=middle+1;else high=middle;
        }
        return -1;
    }
    // First position >= target, or Length. Equality does not stop the loop.
    public static int LowerBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,false,out probes);
    // First position > target, or Length.
    public static int UpperBound<T>(T[] values,T target,IComparer<T> comparer,out long probes)
        => Bound(values,target,comparer,true,out probes);
    private static int Bound<T>(T[] values,T target,IComparer<T> comparer,bool upper,out long probes)
    {
        ArgumentNullException.ThrowIfNull(values);ArgumentNullException.ThrowIfNull(comparer);
        probes=0;int low=0,high=values.Length;
        while(low<high)
        {
            int middle=low+(high-low)/2;
            probes++;int order=comparer.Compare(values[middle],target);
            if(order<0 || upper && order==0)low=middle+1;
            else high=middle;
        }
        return low;
    }
}


At each step, sortedness excludes one side. A smaller value at middle moves low to middle+1; a larger value moves high to middle. Excluding the checked middle ensures progress even in a singleton range. The nonnegative measure high−low strictly decreases. At empty-range exit, no candidate remains. The safe midpoint expression low+(high−low)/2 avoids adding two potentially large indexes. Iterative stack/auxiliary space is Θ(1); worst-case probes are at most floor(log₂ n)+1 for n≥1. Empty input costs zero probes. Linked-list positional access does not have the array’s constant-time assumption.

## Trace a Missing Target

```{index} Trace a Missing Target
```

Trace a missing target as well as a successful one. Keep interval boundaries distinct from valid item indexes: high may equal Length, but middle is always inside the nonempty interval.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,300,400};int target=250;
int low=0,high=sorted.Length;
while(low<high)
{
 int middle=low+(high-low)/2;
 Console.WriteLine($"[{low},{high}): middle={middle}, value={sorted[middle]}");
 int order=sorted[middle].CompareTo(target);
 if(order==0){Console.WriteLine($"found={middle}");break;}
 if(order<0)low=middle+1;else high=middle;
}
Console.WriteLine($"remaining range=[{low},{high})");

For the missing target, the interval ends at [2,2). Its position also suggests where a new value belongs, but the any-match method only promises −1 on absence. A boundary-search method exposes an insertion position directly.

## Library Return Values and Workload Costs

```{index} Library Return Values and Workload Costs
```

Do not assume a library search uses the teaching method’s missing sentinel. Array.BinarySearch returns a negative result on absence; bitwise complement `~result` decodes an insertion position. Test result≥0 to detect success.[^1]

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sorted={100,200,300};
foreach(int target in new[]{50,200,250,500})
{
 int result=Array.BinarySearch(sorted,target);
 Console.WriteLine(result>=0?$"target={target}, found={result}":$"target={target}, raw={result}, insertion={~result}");
}

For q lookups, scanning costs up to Θ(qn). Sorting once and searching costs Θ(n log n+q log n), assuming a comparison sort and static data. Updates can invalidate the sorted snapshot or require maintenance. Sorting merely to answer one query may add work; an existing sorted array can support binary search even for one query. Dictionary exact-key lookup needs its own duplicate and update policy; ordered ranges and all matches are different requirements.

## Choose a Library Contract

```{index} Choose a Library Contract
```

Use the library operation that matches mutation and tie requirements. Array.Sort changes the array and does not guarantee stable ties. OrderBy followed by ToArray materializes a stable sorted array; the input array order remains unchanged.[^1][^2]

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var invoices=new[]{("A",200),("B",100),("C",200)};
var report=invoices.OrderBy(x=>x.Item2).ToArray();
Console.WriteLine($"arrival={string.Join(",",invoices.Select(x=>x.Item1))}");
Console.WriteLine($"report={string.Join(",",report.Select(x=>x.Item1))}");
int[] amounts={300,100,200};Array.Sort(amounts);
Console.WriteLine(string.Join(",",amounts));

## Practice: Prepare a Report

```{index} sorting; report preparation
```


Sort a copy of invoice amounts, preserving the original arrival order. Search the sorted report for 200, report absence for 999, and test an empty array. Explain why a duplicate search returns any match rather than the first arrival.


In [ ]:
using System;
using System.Linq;

int[] amounts = {300, 200, 100, 200};
// Your code starts here.
/* TODO: prepare and search a sorted copy. */
// Your code ends here.


In [1]:
// Solution
using System;
using System.Linq;

int[] amounts = {300, 200, 100, 200};
int[] report = amounts.ToArray();
Array.Sort(report);
Console.WriteLine("arrival=" + string.Join(",", amounts));
Console.WriteLine("report=" + string.Join(",", report));
foreach (int target in new[] {200, 999})
{
    int result = Array.BinarySearch(report, target);
    Console.WriteLine(result >= 0 ? $"found {target}" : $"missing {target}");
}
Console.WriteLine("empty missing=" + (Array.BinarySearch(Array.Empty<int>(), 200) < 0));


arrival=300,200,100,200
report=100,200,200,300
found 200
missing 999
empty missing=True


```{rubric} Footnotes
```
[^1]: `Array.BinarySearch` requires the same ordering used to sort and returns a negative insertion-position encoding on absence; test the sign before indexing.
[^2]: `Array.Sort` mutates its array and does not guarantee stable ties. LINQ `OrderBy` is stable; `ToArray` materializes its result. A shallow copy preserves sequence order, not independent copies of mutable objects.
